# Bronze Kafka Validation

This notebook checks the unified Bronze Delta table at `s3a://bronze/kafka`
and validates that Kafka events are landing as expected.


## Imports and Spark session
Build a Spark session with S3A + Delta settings for MinIO-backed storage.


In [12]:
import os
from pyspark.sql import SparkSession


In [13]:
def build_spark():
    """Create a SparkSession configured for MinIO (S3A) and Delta."""
    app_name = os.getenv('SPARK_APP_NAME', 'bronze-kafka-check')
    return (
        SparkSession.builder
        .appName(app_name)
        .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
        .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
        .config('spark.delta.logStore.class', 'org.apache.spark.sql.delta.storage.S3SingleDriverLogStore')
        .config('spark.hadoop.fs.s3a.endpoint', os.getenv('DATA_LAKE_ENDPOINT', 'http://minio:9000'))
        .config('spark.hadoop.fs.s3a.access.key', os.getenv('DATA_LAKE_ACCESS_KEY_ID', 'minioadmin'))
        .config('spark.hadoop.fs.s3a.secret.key', os.getenv('DATA_LAKE_SECRET_ACCESS_KEY', 'minioadmin'))
        .config('spark.hadoop.fs.s3a.path.style.access', 'true')
        .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
        .getOrCreate()
    )


## Bronze readers
Helper functions for loading and inspecting the Bronze Delta table.


In [14]:
def get_bronze_path():
    """Resolve Bronze path from env or default."""
    return os.getenv('BRONZE_STREAM_PATH', 's3a://bronze/kafka')

def load_bronze_df(spark):
    """Load the Bronze Delta table into a DataFrame."""
    return spark.read.format('delta').load(get_bronze_path())

def show_samples(df, limit=10):
    """Display a sample of Bronze records."""
    df.show(limit, truncate=False)

def show_counts(df):
    """Show total record count."""
    print(f'bronze count: {df.count()}')

def show_record_type_counts(df):
    """Show counts by record_type."""
    (df.groupBy('record_type').count().orderBy('record_type')).show(truncate=False)

def show_kafka_partition_counts(df):
    """Show counts by Kafka partition."""
    (df.groupBy('kafka_partition').count().orderBy('kafka_partition')).show(truncate=False)

def show_file_counts(df, limit=20):
    """Show record counts per Delta file."""
    (df.groupBy('_metadata.file_path')
       .count()
       .orderBy('count', ascending=False)
       .show(limit, truncate=False))


## Run the checks
Start Spark, load Bronze, and inspect counts/samples.


In [15]:
spark = build_spark()
print(f'Spark version: {spark.version}')


Spark version: 4.0.1


In [16]:
bronze_df = load_bronze_df(spark)


In [17]:
show_counts(bronze_df)


[Stage 26:====================================================>   (47 + 3) / 50]

bronze count: 118


In [18]:
show_record_type_counts(bronze_df)


[Stage 32:==============================================>           (4 + 1) / 5]

+-----------+-----+
|record_type|count|
+-----------+-----+
|review     |118  |
+-----------+-----+



In [19]:
show_kafka_partition_counts(bronze_df)


[Stage 37:===========>                                              (1 + 4) / 5]

+---------------+-----+
|kafka_partition|count|
+---------------+-----+
|0              |41   |
|1              |44   |
|2              |33   |
+---------------+-----+



In [20]:
show_samples(bronze_df, limit=5)


+-----------+---------------+---------------+------------+-----------------------+-----------------------+----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

## File-level counts
Check how many records landed in each Delta file.


In [21]:
show_file_counts(bronze_df, limit=20)


[Stage 46:===========>                                              (1 + 4) / 5]

+--------------------------------------------------------------------------------------+-----+
|file_path                                                                             |count|
+--------------------------------------------------------------------------------------+-----+
|s3a://bronze/kafka/part-00001-9e173c99-9662-49aa-addd-854686802f35-c000.snappy.parquet|7    |
|s3a://bronze/kafka/part-00000-a9c0371c-ab6a-4448-8d58-86334b6cc9ce-c000.snappy.parquet|6    |
|s3a://bronze/kafka/part-00002-f589cb25-e4f3-4860-8c18-5fb309e1c242-c000.snappy.parquet|5    |
|s3a://bronze/kafka/part-00000-70232f78-5e54-4b8a-8fc7-1ca840fb6e3e-c000.snappy.parquet|5    |
|s3a://bronze/kafka/part-00001-200e2f1b-4a70-41f0-907c-11184eb2d5c0-c000.snappy.parquet|5    |
|s3a://bronze/kafka/part-00000-07624186-5887-4687-9d85-09b731c1bf50-c000.snappy.parquet|4    |
|s3a://bronze/kafka/part-00001-cb0dd313-b2f2-4744-b467-475a24966554-c000.snappy.parquet|4    |
|s3a://bronze/kafka/part-00002-387442f4-8d73-4a31-